# RAG Generation Notebook — Gemini

End-to-end pipeline: student query → **weighted stratified** S3 Vector retrieval → Gemini generation.

| Step | Description |
|---|---|
| 1 | Install dependencies |
| 2 | Configuration (Gemini API, persona, S3 Vectors / stratified retrieval) |
| 3 | Project path, S3 client, batched `get_vectors`, query embedder |
| 4 | Stratified retrieval (`query_vectors` per corpus + `format_context`) |
| 5 | Build combined prompt |
| 6 | Gemini inference |
| 7 | Save generation record (optional) |

## 1. Dependencies (run once per environment)

In [1]:
%pip install sentence-transformers boto3 pandas google-genai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Configuration

Edit the cells below to change the query, persona, or retrieval settings.

**Gemini API:** `GEMINI_API_KEY` and **`GEMINI_MODEL`** (e.g. `gemini-2.5-flash`) must be set in the project root **`.env`**. These control the **chat model**, not retrieval.

**Retrieval (S3 Vectors):** weighted stratified `query_vectors` uses `RAG_VECTOR_BUCKET`, `RAG_VECTOR_INDEX`, `VECTORS_AWS_*` credentials, `EMBED_MODEL`, **`GEMINI_TOP_K`** (number of `gemini_annotated` chunks to fetch — not the Gemini model id), `RAG_MAX_CONTEXT_CHARS`, `RAG_GET_VECTORS_BATCH`, and optional `RAG_SOURCE_FILE_*` overrides per corpus.

In [30]:
import os
from pathlib import Path

from dotenv import load_dotenv


def _repo_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    return Path.cwd()


load_dotenv(_repo_root() / ".env", override=False)

# ── Student query ────────────────────────────────────────────────────────────
USER_QUERY = "How do I express polite requests in Japanese?"

# ── Gemini — from .env (or shell): GEMINI_API_KEY, GEMINI_MODEL ──────────────
GEMINI_MODEL   = os.environ.get("GEMINI_MODEL", "").strip()
MAX_TOKENS     = int(os.environ.get("GENERATION_MAX_TOKENS", "2048"))
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "").strip()

if not GEMINI_MODEL:
    raise EnvironmentError(
        "GEMINI_MODEL is not set. Add it to the project root .env file "
        '(e.g. GEMINI_MODEL=gemini-2.5-flash) or export it in your shell.'
    )
if not GEMINI_API_KEY:
    raise EnvironmentError(
        "GEMINI_API_KEY is not set. Add it to the project root .env file "
        "or export it in your shell before running."
    )

# ── Tutor persona ─────────────────────────────────────────────────────────────
TUTOR_PERSONA = (
    "You are an encouraging and knowledgeable Japanese language tutor "
    "specialising in domain-specific translation. "
    "You explain grammar and vocabulary clearly, give concrete examples, "
    "and always keep answers practical and supportive. "
    "When the student asks about translation, prefer the most natural "
    "Japanese phrasing and point out any register or politeness considerations."
)

# ── S3 Vectors — stratified retrieval (see rag/query_pipeline.ipynb) ───────────
SOURCE_FILE_ENG_JAP = os.environ.get(
    "RAG_SOURCE_FILE_ENG_JAP", "eng_jap_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GEMINI = os.environ.get(
    "RAG_SOURCE_FILE_GEMINI", "gemini_annotated_chunks_embedded_full.jsonl"
)
SOURCE_FILE_GRAMMAR = os.environ.get(
    "RAG_SOURCE_FILE_GRAMMAR", "grammar_chunks_embedded_full.jsonl"
)
SOURCE_FILE_STYLE = os.environ.get(
    "RAG_SOURCE_FILE_STYLE", "style_guide_chunks_embedded_full.jsonl"
)

# Hits to pull from gemini_annotated stratum only (not GEMINI_MODEL).
GEMINI_TOP_K = int(os.environ.get("GEMINI_TOP_K", "2"))
MAX_CONTEXT_CHARS = int(os.environ.get("RAG_MAX_CONTEXT_CHARS", "12000"))
GET_VECTORS_BATCH = int(os.environ.get("RAG_GET_VECTORS_BATCH", "50"))
EMBED_MODEL = os.environ.get("EMBED_MODEL", "intfloat/multilingual-e5-small")

VECTOR_BUCKET = os.environ.get("RAG_VECTOR_BUCKET", "is469-genai-grp-project")
INDEX_NAME = os.environ.get("RAG_VECTOR_INDEX", "rag-vector-2")
REGION = os.environ.get("VECTORS_AWS_DEFAULT_REGION", "ap-southeast-1")

print(f"Query  : {USER_QUERY}")
print(f"Model  : {GEMINI_MODEL}")
print(f"Bucket : {VECTOR_BUCKET}  |  Index: {INDEX_NAME}  |  Region: {REGION}")
print(
    f"Retrieval: stratified (eng_jap=3, gemini_annotated={GEMINI_TOP_K}, grammar=1, style=1) | "
    f"get_vectors batch={GET_VECTORS_BATCH} | context cap={MAX_CONTEXT_CHARS} | embed={EMBED_MODEL}"
)

Query  : How do I express polite requests in Japanese?
Model  : gemini-2.5-flash
Bucket : is469-genai-grp-project  |  Index: rag-vector-2  |  Region: ap-southeast-1


## 3. Retrieval setup (S3 Vectors)

Project path, types used for hits, S3 client with batched `get_vectors`, and the query embedder (same E5 `query: ` convention as `S3VectorsRAGRetriever`). Chunk text for the prompt comes **only** from vector index metadata — no local `kb/` JSONL reads.

In [24]:
import sys


def project_root() -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    raise RuntimeError("Could not find project root (folder with src/retrieval). cd to the repo root or rag/.")


_ROOT = project_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.retrieval.s3_vectors_rag import RetrievedChunk, format_context
from src.utils.aws_profiles import s3vectors_client

vectors_client = s3vectors_client(region_name=REGION)


def _text_from_vector_metadata(meta: dict) -> str:
    if not meta:
        return ""
    raw = meta.get("text") or meta.get("chunk_text") or meta.get("content")
    if raw is None:
        return ""
    return str(raw).strip()


def resolve_chunks_from_s3(keys: list[str]) -> dict[str, str]:
    if not keys:
        return {}
    uniq = list(dict.fromkeys(k for k in keys if k))
    result: dict[str, str] = {}
    batch = max(1, int(GET_VECTORS_BATCH))
    for i in range(0, len(uniq), batch):
        chunk_keys = uniq[i : i + batch]
        response = vectors_client.get_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=INDEX_NAME,
            keys=chunk_keys,
            returnData=False,
            returnMetadata=True,
        )
        for vec in response.get("vectors", []):
            key = vec.get("key")
            text = _text_from_vector_metadata(vec.get("metadata") or {})
            if key and text:
                result[key] = text
    return result


print(f"Project root : {_ROOT}")
print("S3 Vectors client + batched get_vectors helpers ready.")

Project root : c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469
KB dir       : c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb (exists=True)
Retriever ready (encoder loads lazily on first retrieve).


In [25]:
_embedder = None


def get_embedder():
    global _embedder
    if _embedder is None:
        import torch
        from sentence_transformers import SentenceTransformer

        device = os.environ.get("RAG_EMBED_DEVICE", "cpu")
        if device == "cuda" and not torch.cuda.is_available():
            device = "cpu"
        _embedder = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=device)
        _embedder.max_seq_length = int(
            os.environ.get("MAX_SEQ_LENGTH", "512" if device == "cpu" else "1024")
        )
    return _embedder


def encode_query_vector(query_en: str) -> list[float]:
    model = get_embedder()
    text = query_en if query_en.lower().startswith("query:") else f"query: {query_en}"
    vec = model.encode(text, convert_to_numpy=True, show_progress_bar=False)
    return vec.astype("float32").tolist()


print("Query embedder ready (loads on first encode_query_vector).")

✓ s3vectors_client ready.


## 4. Weighted stratified retrieval

Four filtered `query_vectors` calls (parallel), merge order eng_jap → gemini_annotated → grammar → style_guide, S3-only text, then `format_context` → **`context`** for the prompt below.

In [26]:
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

S3_TEXT_MISSING_PREFIX = "(no chunk text in S3 vector metadata for key="


def _vectors_response_to_chunks(items: list) -> list[RetrievedChunk]:
    out: list[RetrievedChunk] = []
    for item in items:
        key = str(item.get("key", ""))
        distance = item.get("distance")
        if distance is not None:
            try:
                distance = float(distance)
            except (TypeError, ValueError):
                distance = None
        meta = item.get("metadata") or {}
        source_file = str(meta.get("source_file", ""))
        try:
            source_line = int(meta.get("source_line", -1))
        except (TypeError, ValueError):
            source_line = -1
        text = _text_from_vector_metadata(meta)
        out.append(
            RetrievedChunk(
                key=key,
                distance=distance,
                source_file=source_file,
                source_line=source_line,
                text=text,
            )
        )
    return out


STRATA_SPECS = [
    {"name": "eng_jap", "source_file": SOURCE_FILE_ENG_JAP, "top_k": 3},
    {"name": "gemini_annotated", "source_file": SOURCE_FILE_GEMINI, "top_k": GEMINI_TOP_K},
    {"name": "grammar", "source_file": SOURCE_FILE_GRAMMAR, "top_k": 1},
    {"name": "style_guide", "source_file": SOURCE_FILE_STYLE, "top_k": 1},
]

STRATA_MERGE_ORDER = ["eng_jap", "gemini_annotated", "grammar", "style_guide"]


def _query_one_stratum(spec: dict) -> tuple[str, list[RetrievedChunk]]:
    name = spec["name"]
    sf = spec["source_file"]
    k = int(spec["top_k"])
    flt = {"source_file": {"$eq": sf}}
    try:
        resp = vectors_client.query_vectors(
            vectorBucketName=VECTOR_BUCKET,
            indexName=INDEX_NAME,
            topK=k,
            queryVector={"float32": qvec},
            returnMetadata=True,
            returnDistance=True,
            filter=flt,
        )
    except Exception as e:
        print(f"[{name}] query_vectors failed ({sf!r}): {e}")
        return name, []
    vecs = resp.get("vectors") or []
    if not vecs:
        print(f"[{name}] warning: 0 hits for source_file={sf!r} — check SOURCE_FILE_* / unfiltered query sample.")
    chs = _vectors_response_to_chunks(vecs)
    print(f"[{name}] retrieved {len(chs)} chunk(s) (requested top_k={k}).")
    return name, chs


qvec = encode_query_vector(USER_QUERY)

with ThreadPoolExecutor(max_workers=4) as ex:
    stratum_results = list(ex.map(_query_one_stratum, STRATA_SPECS))

by_name = dict(stratum_results)
chunks: list[RetrievedChunk] = []
chunk_stratum: dict[str, str] = {}
seen_keys: set[str] = set()
for nm in STRATA_MERGE_ORDER:
    for c in by_name.get(nm, []):
        if c.key and c.key not in seen_keys:
            seen_keys.add(c.key)
            chunks.append(c)
            chunk_stratum[c.key] = nm

keys_missing_text = [c.key for c in chunks if c.key and not (c.text or "").strip()]
if keys_missing_text:
    print(f"\nFetching text for {len(keys_missing_text)} chunk(s) via get_vectors...")
    s3_text_map = resolve_chunks_from_s3(keys_missing_text)
    for chunk in chunks:
        if chunk.key and not (chunk.text or "").strip():
            chunk.text = s3_text_map.get(chunk.key, "")

for chunk in chunks:
    if chunk.key and not (chunk.text or "").strip():
        chunk.text = (
            f"{S3_TEXT_MISSING_PREFIX}{chunk.key}; "
            f"source_file={chunk.source_file} line={chunk.source_line})"
        )

context = format_context(chunks, max_chars=MAX_CONTEXT_CHARS)

print(f"\n=== Retrieved {len(chunks)} chunk(s); context length {len(context)} chars ===")

_preview = 320
_rows: list[dict] = []
for i, c in enumerate(chunks, start=1):
    t = c.text or ""
    ok = bool(t.strip()) and not t.startswith(S3_TEXT_MISSING_PREFIX)
    preview = t if len(t) <= _preview else f"{t[: _preview - 1]}…"
    _rows.append(
        {
            "rank": i,
            "stratum": chunk_stratum.get(c.key, ""),
            "distance": round(float(c.distance), 6) if c.distance is not None else None,
            "key": c.key,
            "source_file": c.source_file,
            "source_line": c.source_line,
            "resolved": ok,
            "text_preview": preview,
            "chunk_text": t,
        }
    )

hits_df = pd.DataFrame(_rows)
print("\n=== Retrieval hits ===")
hits_df[["rank", "stratum", "distance", "key", "source_file", "source_line", "resolved", "text_preview"]]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector pool : 94 hits (top 100); in source lines 1–100: 3; using 3.

=== Retrieval hits ===


,rank,distance,source_file,source_line,resolved,text_preview
0,1,0.109273,eng_jap_chunks_embedded_full.jsonl,79,True,[1] EN: I'm tired.\n[1] JA: 私は、疲れています。\n\n[2] ...
1,2,0.119010,eng_jap_chunks_embedded_full.jsonl,82,True,[1] EN: I must admit that I snore.\n[1] JA: いび...
2,3,0.119370,eng_jap_chunks_embedded_full.jsonl,80,True,"[1] EN: The check, please.\n[1] JA: 勘定を頼むよ。\n\..."


## 5. Build combined prompt

The system instruction holds the **tutor persona**.  
The user turn holds the **retrieved reference material** + the **student question**.

In [31]:
def build_user_message(query: str, retrieved_context: str) -> str:
    if retrieved_context.strip():
        ctx_block = (
            "### Retrieved reference material\n"
            "The following excerpts come from the project knowledge base "
            "(glossary, translation memory, grammar notes, style guide). "
            "Use them to ground your answer.\n\n"
            f"{retrieved_context}\n\n"
            "---\n"
        )
    else:
        ctx_block = ""  # graceful fallback when retrieval returns nothing
    return ctx_block + f"### Student question\n{query}"


user_message = build_user_message(USER_QUERY, context)

print("=== System instruction (persona) ===")
print(TUTOR_PERSONA)
print("\n=== User message (first 1200 chars) ===")
print(user_message[:1200], "..." if len(user_message) > 1200 else "")

=== System instruction (persona) ===
You are an encouraging and knowledgeable Japanese language tutor specialising in domain-specific translation. You explain grammar and vocabulary clearly, give concrete examples, and always keep answers practical and supportive. When the student asks about translation, prefer the most natural Japanese phrasing and point out any register or politeness considerations.

=== User message (first 1200 chars) ===
### Retrieved reference material
The following excerpts come from the project knowledge base (glossary, translation memory, grammar notes, style guide). Use them to ground your answer.

[eng_jap_chunks_embedded_full.jsonl L79]
[1] EN: I'm tired.
[1] JA: 私は、疲れています。

[2] EN: Who wants some hot chocolate?
[2] JA: ホットチョコレート欲しい人ー？

[3] EN: Speak more slowly, please!
[3] JA: もっとゆっくり話してください！

[4] EN: When do we arrive?
[4] JA: いつ着くの？

[5] EN: The check, please.
[5] JA: 勘定お願いします。

[6] EN: The check, please.
[6] JA: 勘定書をお願いします。

[7] EN: The check, please.
[

## 6. Gemini inference

Uses the `google-genai` SDK (`google.genai`).  
The persona is passed as `system_instruction`; the combined context + query is the user turn.

In [34]:
from google import genai
from google.genai import types
from IPython.display import Markdown, display

client = genai.Client(api_key=GEMINI_API_KEY)

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents=user_message,
    config=types.GenerateContentConfig(
        system_instruction=TUTOR_PERSONA,
        max_output_tokens=MAX_TOKENS,
    ),
)

response_text = response.text

# Usage metadata (not all models populate every field)
usage = response.usage_metadata
print(f"Model          : {GEMINI_MODEL}")
print(f"Input tokens   : {getattr(usage, 'prompt_token_count', 'n/a')}")
print(f"Output tokens  : {getattr(usage, 'candidates_token_count', 'n/a')}")
print("\n" + "="*60)
print("TUTOR RESPONSE")
print("="*60 + "\n")

try:
    display(Markdown(response_text))
except Exception:
    print(response_text)

Model          : gemini-2.5-flash
Input tokens   : 748
Output tokens  : 908

TUTOR RESPONSE



That's a fantastic question, and one of the most practical things to learn when starting out in Japanese! Expressing polite requests is essential for smooth communication.

Let's break down how to do this, looking at your reference material and adding some more common patterns.

---

### The Most Common & Versatile Way: 「～てください」

This is your go-to form for making polite requests. It's formed by taking the **te-form (て形)** of a verb and adding **ください (kudasai)**.

*   **Formation:** Verb (て形) + ください
*   **Meaning:** "Please do [verb]"

**Example from your reference:**

*   **EN:** Speak more slowly, please!
*   **JA:** もっとゆっくり**話してください**！ (motto yukkuri **hanashite kudasai**!)
    *   Here, 話す (hanasu - to speak) becomes 話して (hanashite) in its te-form, then ください is added.

**Other common examples:**

*   座る (suwaru - to sit) → **座ってください** (suwatte kudasai) - Please sit down.
*   待つ (matsu - to wait) → **待ってください** (matte kudasai) - Please wait.
*   書く (kaku - to write) → **書いてください** (kaite kudasai) - Please write it.

This form is widely applicable and generally polite enough for most everyday situations, like asking a shop assistant, a classmate, or someone you're speaking to for the first time.

---

### Requesting Items or Services: 「～をお願いします」 or 「～ください」

When you want to ask for a *thing* or a *service*, 「お願いします (onegai shimasu)」 is incredibly useful. 「お願いします」 literally means "I humbly request" or "I ask of you."

*   **Formation 1:** Noun + を (o) + お願いします
*   **Formation 2:** Noun + お願いします (often used informally, or when "を" is implied)
*   **Meaning:** "Please give me [noun]" or "I'd like [noun]"

**Examples from your reference:**

*   **EN:** The check, please.
*   **JA:** 勘定**お願いします**。 (kanjou **onegai shimasu**.)
    *   Here, 勘定 (kanjou - the check/bill) is followed by お願いします.
*   **JA:** 勘定書**をお願いします**。 (kanjousho **wo onegai shimasu**.)
    *   Same meaning, just with the noun 勘定書 (kanjousho - check/bill paper) and the particle を.
*   **JA:** お愛想**お願いします**。 (oaiso **onegai shimasu**.)
    *   お愛想 (oaiso) is another common way to say "the check" in a restaurant, followed by お願いします.

You can also combine a noun directly with **ください** in these contexts, especially for common items:

*   **Formation:** Noun + ください
*   **Meaning:** "Please give me [noun]"

**Example:**

*   お水**ください** (omizu **kudasai**) - Please give me water.
*   コーヒー**ください** (koohii **kudasai**) - Please give me coffee.

This form is direct but generally polite enough when ordering in a casual setting.

---

### More Polite & Indirect Requests: 「～ていただけますか / いただけませんか」

These forms are more formal and indirect, making them sound even more polite. They literally mean "Would you be able to do [verb] for me?" or "Wouldn't you be able to do [verb] for me?"

*   **Formation:** Verb (て形) + いただけますか (itadakemasu ka)
*   **Formation:** Verb (て形) + いただけませんか (itadakemasen ka)
*   **Meaning:** "Could you please [verb] for me?" / "Would